# Linear ODE Discretization Methods

Chapter 20: ODE, SDE, and Continuous Limits of Algorithms · *Mathematics for AI and Machine Learning* by Fuheng Wu

Euler, exact zero-order hold, and Tustin updates for the same stable linear system.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-20/discretization-methods/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

# The demo's defaults: dx/dt = a x with a = -1.2, step h = 0.35, x0 = 1, 45 steps
a, h, x0, K = -1.2, 0.35, 1.0, 45

# Each one-step method multiplies by a factor phi per step
methods = {
    "explicit Euler": 1 + h * a,
    "implicit Euler": 1 / (1 - h * a),
    "trapezoid (Tustin)": (1 + 0.5 * h * a) / (1 - 0.5 * h * a),
    "exact": math.exp(a * h),
}
for name, phi in methods.items():
    print(f"{name:20s} phi = {phi:.6f}   x after {K} steps = {x0 * phi ** K:.3e}")

# ── Global error at t = 2 as h shrinks: orders 1, 1, 2 ──────────────────────
print("\n   h      explicit     implicit     trapezoid")
for hh in (0.4, 0.2, 0.1, 0.05):
    n = round(2 / hh)
    exact = x0 * math.exp(2 * a)
    errs = [abs(x0 * p ** n - exact) for p in (1 + hh * a, 1 / (1 - hh * a), (1 + 0.5 * hh * a) / (1 - 0.5 * hh * a))]
    print(f"{hh:5.2f}   {errs[0]:.3e}    {errs[1]:.3e}    {errs[2]:.3e}")
# Halving h halves the Euler errors and quarters the trapezoid error.

# ── The same methods as code, on a 2-state linear system x' = A x ───────────
A = torch.tensor([[-1.2, 0.5], [0.0, -3.0]], dtype=torch.float64)
I = torch.eye(2, dtype=torch.float64)
x = torch.tensor([1.0, 1.0], dtype=torch.float64)
step = {"explicit": I + h * A, "implicit": torch.linalg.inv(I - h * A),
        "trapezoid": torch.linalg.solve(I - 0.5 * h * A, I + 0.5 * h * A)}
exact = torch.linalg.matrix_exp(A * h * 10) @ x
for name, M in step.items():
    print(f"2D {name:9s} after 10 steps: {[round(v, 5) for v in (torch.linalg.matrix_power(M, 10) @ x).tolist()]}")
print("2D exact                :", [round(v, 5) for v in exact.tolist()])

explicit Euler       phi = 0.580000   x after 45 steps = 2.261e-11
implicit Euler       phi = 0.704225   x after 45 steps = 1.403e-07
trapezoid (Tustin)   phi = 0.652893   x after 45 steps = 4.655e-09
exact                phi = 0.657047   x after 45 steps = 6.192e-09

   h      explicit     implicit     trapezoid
 0.40   5.270e-02    5.011e-02    4.229e-03
 0.20   2.643e-02    2.564e-02    1.048e-03
 0.10   1.316e-02    1.295e-02    2.615e-04
 0.05   6.556e-03    6.504e-03    6.533e-05
2D explicit  after 10 steps: [0.0055, 0.0]
2D implicit  after 10 steps: [0.03812, 0.00076]
2D trapezoid after 10 steps: [0.01798, 1e-05]
2D exact                : [0.01915, 3e-05]
